# Parse PDFs with PyMuPDF

This notebook recursively finds PDF files inside `data/` (across subfolders such as `ero-reliability-risk-priorities-reports/`, `event_analysis_reports/`, and `state_of_reliability_reports/`), parses them with the open-source `PyMuPDF` package, and saves the extracted text as JSON.

Each PDF's parent folder name is captured as its `category`, so downstream steps can filter or group by report type.

## Setup

Install the dependencies from `requirements.txt` before running the notebook:

```bash
pip install -r requirements.txt
```

Expected layout (PDFs live in category subfolders under `data/`):

```
data_pipeline/
└── data/
    ├── ero-reliability-risk-priorities-reports/*.pdf
    ├── event_analysis_reports/*.pdf
    └── state_of_reliability_reports/*.pdf
```

The notebook works whether you launch Jupyter from the repo root or from inside `data_pipeline/`.

In [33]:
from pathlib import Path
import json

import pandas as pd
from tqdm.auto import tqdm
import fitz  # PyMuPDF

In [34]:
CANDIDATE_DATA_DIRS = [Path("data_pipeline/data"), Path("data")]
DATA_DIR = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), CANDIDATE_DATA_DIRS[0])

CANDIDATE_OUTPUT_DIRS = [Path("data_pipeline"), Path(".")]
OUTPUT_DIR = next((p for p in CANDIDATE_OUTPUT_DIRS if p.exists()), CANDIDATE_OUTPUT_DIRS[-1])
OUTPUT_PATH = OUTPUT_DIR / "parsed_pdfs.json"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Reading PDFs from: {DATA_DIR.resolve()}")
print(f"Saving parsed output to: {OUTPUT_PATH.resolve()}")

categories = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
print(f"Found {len(categories)} category folder(s): {categories}")

Reading PDFs from: D:\OneDrive\Documents\Hackathons\SCSP\Project_2\GridSync\data_pipeline\data
Saving parsed output to: D:\OneDrive\Documents\Hackathons\SCSP\Project_2\GridSync\data_pipeline\parsed_pdfs.json
Found 3 category folder(s): ['ero-reliability-risk-priorities-reports', 'event_analysis_reports', 'state_of_reliability_reports']


In [35]:
pdf_files = sorted(DATA_DIR.rglob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s) across all subfolders.\n")

from collections import Counter

counts_by_category = Counter(
    pdf.relative_to(DATA_DIR).parts[0] if pdf.relative_to(DATA_DIR).parts[:-1] else "(root)"
    for pdf in pdf_files
)
for category, count in sorted(counts_by_category.items()):
    print(f"  {category}: {count} PDF(s)")

Found 61 PDF file(s) across all subfolders.

  ero-reliability-risk-priorities-reports: 12 PDF(s)
  event_analysis_reports: 30 PDF(s)
  state_of_reliability_reports: 19 PDF(s)


In [36]:
import re

# pymupdf4llm wraps these markers in **bold** and appends <br> tags.
# We remove only the three marker lines themselves and keep any
# content that sits between Start and End, e.g.:
#
#   BEFORE                                  AFTER
#   ──────────────────────────────────────  ─────────────────────────────────
#   **==> picture [612 x 55] intentionally omitted <==**   (removed)
#   **----- Start of picture text -----**<br>              (removed)
#   NERC | Report | November 2019<br>10<br>                NERC | Report | November 2019<br>10<br>
#   **----- End of picture text -----**<br>                (removed)

_PICTURE_RE = re.compile(
    # 1. Picture-omission line (** bold optional, any WxH dimensions)
    r"^\**==> picture \[\d+ x \d+\] intentionally omitted <==\**\n?"
    # 2. Start-of-picture-text marker line only
    r"|^\**-{5} Start of picture text -{5}\**<br>\n?"
    # 3. End-of-picture-text marker — may appear mid-line after content,
    #    so match from the marker to end-of-line (including trailing <br>)
    r"|\**-{5} End of picture text -{5}\**<br>",
    re.IGNORECASE | re.MULTILINE,
)

def _clean_text(text: str) -> str:
    """Remove pymupdf4llm image-placeholder marker lines and tidy whitespace."""
    text = _PICTURE_RE.sub("", text)
    text = re.sub(r"\n{3,}", "\n\n", text)   # collapse runs of blank lines
    return text.strip()

In [37]:
# I want to loop over pdf files in event_analysis_reports and convert each pdf to text and then parse it to the Vector database (you can create a dummy vector database parsing function)


In [38]:
from pathlib import Path
import pymupdf4llm

# DATA_DIR = Path("your/data/dir")  # change this

pdf_files = sorted(DATA_DIR.rglob("*.pdf"))

# Take only the first PDF
pdf_files = pdf_files[:1]

print(f"Found {len(pdf_files)} PDF file(s).\n")

for pdf in pdf_files:
    print(f"Converting: {pdf}\n")

    md_text = _clean_text(pymupdf4llm.to_markdown(str(pdf)))

    print("----- MARKDOWN OUTPUT -----\n")
    print(md_text)

Found 1 PDF file(s).

Converting: data\ero-reliability-risk-priorities-reports\2019 ERO Reliability Risk Priorities Report November 2019.pdf

----- MARKDOWN OUTPUT -----

# **2019 ERO Reliability Risk Priorities Report** 

November 2019 

**NERC | Report Title | Report Date I** 

## **Table of Contents** 

Preface ........................................................................................................................................................................... iii **Executive Summary** ................................................................................................................................................. 5 Common Themes and Emerging Trends ...................................................................................................................... 6 **Background and Introduction** ............................................................................................................................. 8 Inputs to the Risk Prof

In [39]:
_HEADER_RE = re.compile(r"^(#{1,6})\s+(.+)$", re.MULTILINE)
_MD_BOLD_RE = re.compile(r"\*{1,2}(.+?)\*{1,2}")   # strip **bold** / *italic*


def chunk_by_headers(md_text: str, source: str = "", min_body_chars: int = 0) -> list[dict]:
    """Split cleaned markdown text into chunks at every header boundary.

    Each chunk spans from one header (inclusive) to the line just before
    the next header of any level.  An optional preamble chunk is created
    for any text that precedes the first header.

    Parameters
    ----------
    md_text : str
        Cleaned markdown string, e.g. the output of ``_clean_text()``.
    source : str
        Source identifier (filename or path) stored in each chunk's metadata.
    min_body_chars : int
        Skip chunks whose body text is shorter than this many characters.
        Useful for filtering out near-empty section stubs (default 0 = keep all).

    Returns
    -------
    list[dict]
        One dict per chunk with keys:

        chunk_index  – sequential index (0-based)
        level        – heading depth (0 = preamble, 1 = #, 2 = ##, …)
        header       – raw header line including # prefix and any markdown
        title        – clean plain-text title (bold/italic markers stripped)
        body         – text beneath the header, up to the next header
        content      – header line + body (ready to embed as-is)
        source       – value of the ``source`` parameter

    Example
    -------
    ::

        md = _clean_text(pymupdf4llm.to_markdown("report.pdf"))
        chunks = chunk_by_headers(md, source="report.pdf")
        for c in chunks:
            print(c["level"], c["title"], "–", len(c["body"]), "chars")
    """
    matches = list(_HEADER_RE.finditer(md_text))
    chunks: list[dict] = []

    def _plain(text: str) -> str:
        """Strip markdown bold/italic to get plain title text."""
        return _MD_BOLD_RE.sub(r"\1", text).strip()

    def _make_chunk(level, raw_header, title, body, source):
        body = body.strip()
        if len(body) < min_body_chars:
            return None
        return {
            "chunk_index": len(chunks),
            "level":       level,
            "header":      raw_header,
            "title":       title,
            "body":        body,
            "content":     (raw_header + "\n" + body).strip() if raw_header else body,
            "source":      source,
        }

    # --- preamble (text before the first header) ----------------------------
    if matches:
        preamble = md_text[: matches[0].start()].strip()
    else:
        preamble = md_text.strip()

    if preamble:
        chunk = _make_chunk(0, "", "(Preamble)", preamble, source)
        if chunk:
            chunks.append(chunk)

    # --- one chunk per header -----------------------------------------------
    for i, m in enumerate(matches):
        level     = len(m.group(1))          # count of '#' chars
        raw_hdr   = m.group(0)               # e.g. "## **Executive Summary**"
        title     = _plain(m.group(2))       # "Executive Summary"

        body_start = m.end()
        body_end   = matches[i + 1].start() if i + 1 < len(matches) else len(md_text)
        body       = md_text[body_start:body_end]

        chunk = _make_chunk(level, raw_hdr, title, body, source)
        if chunk:
            chunks.append(chunk)

    return chunks

In [40]:
def merge_small_chunks(chunks: list[dict], min_chars: int = 1000) -> list[dict]:
    """Merge consecutive chunks until the accumulated content reaches *min_chars*.

    Chunks are merged in order.  When the running total of body text hits
    ``min_chars`` the group is finalised and a new group starts.  Any
    leftover chunks at the end are emitted as-is (they may still be below
    ``min_chars`` if the document simply has no more content).

    Merged chunk fields
    -------------------
    title   – titles of all merged chunks joined with  " | "
    header  – header of the first chunk in the group
    level   – level of the first chunk in the group
    body    – bodies joined with "\\n\\n"
    content – headers + bodies joined with "\\n\\n"
    source  – source of the first chunk in the group

    Parameters
    ----------
    chunks : list[dict]
        Output of ``chunk_by_headers()``.
    min_chars : int
        Minimum body character count before a group is closed (default 1000).

    Returns
    -------
    list[dict]
        New list of merged chunks, re-indexed from 0.
    """
    if not chunks:
        return []

    merged: list[dict] = []
    group: list[dict] = []
    running_chars = 0

    def _flush(group: list[dict]) -> dict:
        first = group[0]
        title   = " | ".join(c["title"] for c in group if c["title"])
        body    = "\n\n".join(c["body"]    for c in group if c["body"])
        content = "\n\n".join(c["content"] for c in group if c["content"])
        return {
            "chunk_index": len(merged),
            "level":       first["level"],
            "header":      first["header"],
            "title":       title,
            "body":        body,
            "content":     content,
            "source":      first["source"],
        }

    for chunk in chunks:
        group.append(chunk)
        running_chars += len(chunk["body"])

        if running_chars >= min_chars:
            merged.append(_flush(group))
            group = []
            running_chars = 0

    # flush any remaining chunks that didn't reach the threshold
    if group:
        merged.append(_flush(group))

    return merged

In [41]:
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from google import genai as google_genai

# Load GEMINI_API_KEY from .env (works from repo root or data_pipeline/)
for _env in [Path(".env"), Path("../.env")]:
    if _env.exists():
        load_dotenv(_env)
        break

_CONTEXT_PROMPT = """\
You are given an excerpt (chunk) from a NERC (North American Electric Reliability \
Corporation) reliability report. Your task is to write 2-3 sentences that:
  1. Identify the document (report type and approximate year if discernible). Include this everytime. 
  2. Describe what topic this specific chunk covers and how it fits within the \
overall document structure.
Both 1 and 2 are must.

Output ONLY the context sentences — no bullet points, no labels, no preamble.

<document>
{doc_preview}
</document>

<chunk>
{chunk_content}
</chunk>
"""


def generate_chunk_contexts(
    chunks: list[dict],
    full_doc: str,
    model: str | None = None,
    max_doc_chars: int = 12_000,
    max_workers: int = 10,
) -> list[dict]:
    """Prepend a Gemini-generated situating context to every chunk.

    Calls Gemini in parallel using a ``ThreadPoolExecutor`` with up to
    ``max_workers`` concurrent threads (default 10), so all chunks are
    dispatched at once instead of sequentially.

    Parameters
    ----------
    chunks : list[dict]
        Output of ``merge_small_chunks()`` (or ``chunk_by_headers()``).
    full_doc : str
        Full cleaned markdown text of the document the chunks came from.
    model : str, optional
        Gemini model ID.  Defaults to the ``GEMINI_MODEL`` env var or
        ``"gemini-2.5-flash"``.
    max_doc_chars : int
        Maximum characters of ``full_doc`` sent to the model (default 12 000).
        If longer, the first and last halves are kept with ``[…]`` in between.
    max_workers : int
        Number of parallel threads (default 10).

    Returns
    -------
    list[dict]
        A new list of chunk dicts in the original order, each with an added
        ``"context"`` key and ``"content"`` prepended with the generated context.
    """
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise EnvironmentError("GEMINI_API_KEY not set — check your .env file.")

    model = model or os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

    # Build a document preview that fits within max_doc_chars
    if len(full_doc) > max_doc_chars:
        half = max_doc_chars // 2
        doc_preview = full_doc[:half] + "\n\n[… document truncated …]\n\n" + full_doc[-half:]
    else:
        doc_preview = full_doc

    # Each thread gets its own client (google-genai clients are not thread-safe)
    _local = threading.local()

    def _get_client() -> google_genai.Client:
        if not hasattr(_local, "client"):
            _local.client = google_genai.Client(api_key=api_key)
        return _local.client

    def _generate_one(chunk: dict) -> tuple[int, str]:
        """Return (original chunk_index, generated context string)."""
        prompt = _CONTEXT_PROMPT.format(
            doc_preview=doc_preview,
            chunk_content=chunk["content"],
        )
        try:
            response = _get_client().models.generate_content(model=model, contents=prompt)
            return chunk["chunk_index"], response.text.strip()
        except Exception as exc:
            print(f"  [WARN] chunk {chunk['chunk_index']} – context generation failed: {exc}")
            return chunk["chunk_index"], ""

    # Submit all chunks in parallel, collect results keyed by chunk_index
    contexts: dict[int, str] = {}
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(_generate_one, chunk): chunk for chunk in chunks}
        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"Generating contexts ({max_workers} threads)",
        ):
            idx, context = future.result()
            contexts[idx] = context

    # Rebuild list in original order
    enriched: list[dict] = []
    for chunk in chunks:
        context = contexts.get(chunk["chunk_index"], "")
        new_chunk = chunk.copy()
        new_chunk["context"] = context
        new_chunk["content"] = (f"{context}\n\n{chunk['content']}") if context else chunk["content"]
        enriched.append(new_chunk)

    return enriched

In [42]:
import uuid

def attach_metadata(chunks: list[dict], folder: str, filename: str) -> list[dict]:
    """Attach a metadata dict to every chunk in the list.

    A single UUID is generated once per file and shared across all chunks
    that originate from it, so chunks can be grouped or filtered by file
    without re-parsing the filename.

    Parameters
    ----------
    chunks : list[dict]
        Final enriched chunk list (output of ``generate_chunk_contexts``).
    folder : str
        Category folder name, e.g. ``"ero-reliability-risk-priorities-reports"``.
    filename : str
        Bare PDF filename, e.g. ``"2019 ERO Reliability Risk Priorities Report.pdf"``.

    Returns
    -------
    list[dict]
        Same chunks with a ``"metadata"`` key added to each::

            {
                "folder"   : str   # category subfolder name
                "filename" : str   # PDF filename
                "file_id"  : str   # UUID4 – same for every chunk in this file
                "chunk_no" : int   # 0-based serial number within the file
            }
    """
    file_id = str(uuid.uuid4())   # one UUID shared across all chunks of this file

    result: list[dict] = []
    for chunk_no, chunk in enumerate(chunks):
        new_chunk = chunk.copy()
        new_chunk["metadata"] = {
            "folder":    folder,
            "filename":  filename,
            "file_id":   file_id,
            "chunk_no":  chunk_no,
        }
        result.append(new_chunk)
    return result

In [43]:
# ---------------------------------------------------------------------------
# Demo – chunk the first PDF from each category and print all chunk content
# ---------------------------------------------------------------------------

def _first_pdf_markdown(directory: Path) -> tuple[str, str]:
    """Return (cleaned markdown text, filename) for the first PDF in directory."""
    pdfs = sorted(directory.rglob("*.pdf"))
    if not pdfs:
        return "", ""
    path = pdfs[0]
    md = _clean_text(pymupdf4llm.to_markdown(str(path)))
    return md, path.name


ERO_DIR   = DATA_DIR / "ero-reliability-risk-priorities-reports"
EVENT_DIR = DATA_DIR / "event_analysis_reports"
SOR_DIR   = DATA_DIR / "state_of_reliability_reports"

for label, directory in [("ERO", ERO_DIR), ("Event Analysis", EVENT_DIR), ("State of Reliability", SOR_DIR)]:
    md, fname = _first_pdf_markdown(directory)
    if not md:
        print(f"{label}: no PDFs found.\n")
        continue

    raw_chunks    = chunk_by_headers(md, source=fname, min_body_chars=50)
    merged_chunks = merge_small_chunks(raw_chunks, min_chars=1000)
    ctx_chunks    = generate_chunk_contexts(merged_chunks, full_doc=md)
    final_chunks  = attach_metadata(ctx_chunks, folder=directory.name, filename=fname)

    print(f"{'=' * 70}")
    print(f"  {label}  →  {fname}")
    print(f"{'─' * 70}")
    print(f"  Before merging : {len(raw_chunks):>3} chunks  "
          f"| sizes (chars): {[len(c['body']) for c in raw_chunks]}")
    print(f"  After  merging : {len(merged_chunks):>3} chunks  "
          f"| sizes (chars): {[len(c['body']) for c in merged_chunks]}")
    print(f"{'=' * 70}\n")

    for c in final_chunks:
        m   = c["metadata"]
        bar = "#" * c["level"] if c["level"] else "~"
        print(f"┌─ Chunk {m['chunk_no']}  {bar}  \"{c['title']}\"  ({len(c['body'])} chars)")
        print(f"  metadata : folder={m['folder']}  |  file_id={m['file_id']}  |  chunk_no={m['chunk_no']}")
        print(f"             filename={m['filename']}")
        print(f"  ── CONTEXT ──────────────────────────────────────────────────")
        print(f"  {c['context']}")
        print(f"  ── CONTENT ──────────────────────────────────────────────────")
        print(c["content"])
        print(f"└{'─' * 68}\n")
    break
    print()

Generating contexts (10 threads): 100%|██████████| 18/18 [00:09<00:00,  1.83it/s]

  ERO  →  2019 ERO Reliability Risk Priorities Report November 2019.pdf
──────────────────────────────────────────────────────────────────────
  Before merging :  26 chunks  | sizes (chars): [55, 2082, 496, 1912, 3954, 1175, 2551, 3626, 3506, 195, 1365, 464, 4670, 3461, 857, 670, 2807, 2593, 1431, 639, 2245, 1495, 389, 605, 782, 1395]
  After  merging :  18 chunks  | sizes (chars): [2139, 2410, 3954, 1175, 2551, 3626, 3506, 1562, 5136, 3461, 1529, 2807, 2593, 1431, 2886, 1495, 1780, 1395]

┌─ Chunk 0  #  "2019 ERO Reliability Risk Priorities Report | Table of Contents"  (2139 chars)
  metadata : folder=ero-reliability-risk-priorities-reports  |  file_id=5b46acc0-39f7-4f8e-b9bb-afe0e82d0919  |  chunk_no=0
             filename=2019 ERO Reliability Risk Priorities Report November 2019.pdf
  ── CONTEXT ──────────────────────────────────────────────────
  This excerpt is from the 2019 ERO Reliability Risk Priorities Report, published by NERC in November 2019. The chunk specifically present